In [1]:
#1

def is_valid_invitation(invited, graph):

    k = len(invited)

    # Check for conflicts by pairing any two users (u, v) from the invited list
    for i in range(k):
        for j in range(i + 1, k):
            u = invited[i]
            v = invited[j]

            if u in graph.get(v, set()):
                return False

    return True


# Test
if __name__ == "__main__":

    pulse_conflict_graph = {
        'Alice': {'Bob', 'Charlie'}, # Alice has a conflict with Bob and Charlie
        'Bob': {'Alice', 'David'},
        'Charlie': {'Alice'},
        'David': {'Bob'}
    }

    # Test Case 1: No conflicts (Alice and David are not connected)
    invited_group_1 = ['Alice', 'David']
    print(f"Group 1 Validity: {is_valid_invitation(invited_group_1, pulse_conflict_graph)}") # Expected: True

    # Test Case 2: Conflict exists (Alice and Charlie have a conflict)
    invited_group_2 = ['Alice', 'Charlie', 'David']
    print(f"Group 2 Validity: {is_valid_invitation(invited_group_2, pulse_conflict_graph)}") # Expected: False

Group 1 Validity: True
Group 2 Validity: False


In [3]:
#2

def find_max_invitations_exact(graph):
    n = len(graph)
    best_set = set()
    current_set = set()

    def backtrack(current_node):
        nonlocal best_set

        if len(current_set) + (n - current_node) <= len(best_set):
            return

        # Base case
        if current_node == n:
            if len(current_set) > len(best_set):
                best_set = current_set.copy()
            return

        is_conflict = False

        for neighbor in graph[current_node]:
            if neighbor in current_set:
                is_conflict = True
                break

        if not is_conflict:
            current_set.add(current_node)
            backtrack(current_node + 1)
            current_set.remove(current_node)

        backtrack(current_node + 1)

    backtrack(0)

    max_size = len(best_set)
    invited_list = list(best_set)

    return (max_size, invited_list)


# Test Cases

if __name__ == "__main__":

    # Test Case 1: Linear relationship (0-1-2-3)
    # Node 0 conflicts with 1; Node 1 with 0, 2; Node 2 with 1, 3; Node 3 with 2.
    # Expected answer: [0, 2] or [1, 3] (max size 2)
    test_graph_1 = [
        [1],       # Node 0
        [0, 2],    # Node 1
        [1, 3],    # Node 2
        [2]        # Node 3
    ]

    size1, invited1 = find_max_invitations_exact(test_graph_1)
    print("=== Test Case 1 ===")
    print(f"Max Size: {size1}")
    print(f"Invited List: {invited1}\n")

    # Test Case 2: Star shape (one node conflicts with all others)
    # Node 0 conflicts with 1, 2, and 3. Nodes 1, 2, and 3 have no conflicts among themselves.
    # Expected answer: Invite [1, 2, 3] excluding node 0 (max size 3)
    test_graph_2 = [
        [1, 2, 3], # Node 0
        [0],       # Node 1
        [0],       # Node 2
        [0]        # Node 3
    ]

    size2, invited2 = find_max_invitations_exact(test_graph_2)
    print("=== Test Case 2 ===")
    print(f"Max Size: {size2}")
    print(f"Invited List: {invited2}")

=== Test Case 1 ===
Max Size: 2
Invited List: [0, 2]

=== Test Case 2 ===
Max Size: 3
Invited List: [1, 2, 3]


In [4]:
#3
def find_max_invitations_greedy(graph):
    invited_list = []
    available_nodes = set(range(len(graph)))

    while available_nodes:
        min_degree = float('inf')
        best_node = -1

        for node in available_nodes:
            degree = sum(1 for neighbor in graph[node] if neighbor in available_nodes)

            if degree < min_degree:
                min_degree = degree
                best_node = node

        invited_list.append(best_node)

        available_nodes.remove(best_node)
        for neighbor in graph[best_node]:
            if neighbor in available_nodes:
                available_nodes.remove(neighbor)

    size = len(invited_list)
    return (size, invited_list)

# Test
if __name__ == "__main__":


    def find_max_invitations_exact(graph):
        n = len(graph)
        best_set = set()
        current_set = set()

        def backtrack(current_node):
            nonlocal best_set
            if len(current_set) + (n - current_node) <= len(best_set):
                return
            if current_node == n:
                if len(current_set) > len(best_set):
                    best_set = current_set.copy()
                return
            is_conflict = False
            for neighbor in graph[current_node]:
                if neighbor in current_set:
                    is_conflict = True
                    break
            if not is_conflict:
                current_set.add(current_node)
                backtrack(current_node + 1)
                current_set.remove(current_node)
            backtrack(current_node + 1)

        backtrack(0)
        return len(best_set), list(best_set)

    # Comparison Graph: A graph where Greedy might choose sub-optimally
    # Node 0 conflicts with 1, 2
    # Node 1 conflicts with 0, 3, 4
    # Node 2 conflicts with 0, 5, 6
    # Nodes 3, 4, 5, 6 only conflict with 1 and 2
    test_graph = [
        [1, 2],       # 0
        [0, 3, 4],    # 1
        [0, 5, 6],    # 2
        [1],          # 3
        [1],          # 4
        [2],          # 5
        [2]           # 6
    ]

    exact_size, exact_list = find_max_invitations_exact(test_graph)
    greedy_size, greedy_list = find_max_invitations_greedy(test_graph)

    print("Comparison on Small Graph")
    print(f"Exact Solution : Size {exact_size}, List {exact_list}")
    print(f"Greedy Solution: Size {greedy_size}, List {greedy_list}")

    if exact_size == greedy_size:
        print("-> Analysis: Greedy found the optimal solution for this graph.")
    else:
        print("-> Analysis: Greedy fell into a local optimum and found a smaller set.")

=== Comparison on Small Graph ===
Exact Solution : Size 5, List [0, 3, 4, 5, 6]
Greedy Solution: Size 5, List [3, 4, 0, 5, 6]
-> Analysis: Greedy found the optimal solution for this graph.
